# Chapter 01: The Neuron (Linear and Logistic Regression)

Welcome to the second notebook of **Neural Networks from Scratch**!

In this lab, we build and train the simplest neural architectures:
1. **Linear Regression**: A continuous neuron minimizing Mean Squared Error (MSE)
2. **Visualizing the Loss Curve and Line of Best Fit**
3. **Logistic Regression**: A single artificial neuron with Sigmoid activation
4. **Visualizing the 2D Decision Boundary** in feature space
5. **PyTorch Equivalence Verification**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure nnlib is on sys.path
sys.path.insert(0, str(Path.cwd().parents[1] if Path.cwd().name == "01-the-neuron" else Path.cwd()))
from nnlib.linear import LinearRegression, LogisticRegression
from nnlib.activations import sigmoid
from nnlib.device import get_device_name

print("Compute device:", get_device_name())

## 1. Linear Regression on Synthetic 1D Data

Let us generate $N = 100$ data points where the true relationship is:
$$y = 2.5 x + 1.2 + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 0.3^2)$$

In [ ]:
rng = np.random.RandomState(42)
N = 100
X_1d = rng.uniform(-3, 3, size=(N, 1))
true_w = 2.5
true_b = 1.2
noise = rng.randn(N, 1) * 0.4
y_1d = true_w * X_1d + true_b + noise

plt.figure(figsize=(8, 4.5))
plt.scatter(X_1d, y_1d, alpha=0.7, color="#2563eb", label="Noisy Samples")
plt.title("Synthetic 1D Linear Data", fontsize=14)
plt.xlabel("Input Feature x")
plt.ylabel("Target y")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Training Linear Regression with Gradient Descent

In [ ]:
lin_reg = LinearRegression(in_features=1)
lin_reg.fit(X_1d, y_1d, lr=0.05, epochs=100, verbose=True)

print(f"\nLearned weight w: {lin_reg.w[0, 0]:.4f} (True: {true_w})")
print(f"Learned bias   b: {lin_reg.b:.4f} (True: {true_b})")

### Visualizing the Fitted Line and Loss Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Fitted Line Plot
x_grid = np.linspace(-3.5, 3.5, 100).reshape(-1, 1)
y_grid = lin_reg.predict(x_grid)

ax1.scatter(X_1d, y_1d, alpha=0.6, color="#2563eb", label="Data Points")
ax1.plot(x_grid, y_grid, color="#dc2626", linewidth=2.5, label="Fitted Line (y = wx + b)")
ax1.set_title("Linear Regression: Fit to Data", fontsize=13)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Loss Curve
ax2.plot(lin_reg.loss_history, color="#059669", linewidth=2)
ax2.set_title("MSE Loss During Training", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MSE Loss")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Logistic Regression: 2D Binary Classification

Let us create two distinct clusters in 2D space:
- Class 0 centered at $(-1.5, -1.5)$
- Class 1 centered at $(1.5, 1.5)$

In [ ]:
rng = np.random.RandomState(42)
n_pts = 75

# Class 0
X0 = rng.randn(n_pts, 2) * 0.8 + np.array([-1.5, -1.5])
y0 = np.zeros((n_pts, 1))

# Class 1
X1 = rng.randn(n_pts, 2) * 0.8 + np.array([1.5, 1.5])
y1 = np.ones((n_pts, 1))

X_2d = np.vstack([X0, X1])
y_2d = np.vstack([y0, y1])

plt.figure(figsize=(7, 6))
plt.scatter(X0[:, 0], X0[:, 1], color="#2563eb", alpha=0.8, label="Class 0")
plt.scatter(X1[:, 0], X1[:, 1], color="#dc2626", alpha=0.8, label="Class 1")
plt.title("2D Linearly Separable Dataset", fontsize=14)
plt.xlabel("Feature x_1")
plt.ylabel("Feature x_2")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Training the Logistic Neuron

In [ ]:
log_reg = LogisticRegression(in_features=2)
log_reg.fit(X_2d, y_2d, lr=0.2, epochs=250, verbose=True)

accuracy = np.mean(log_reg.predict(X_2d) == y_2d)
print(f"\nTraining Accuracy: {accuracy * 100:.2f}%")

### Visualizing the 2D Decision Boundary & Probability Landscape

The decision boundary occurs where $z = w_1 x_1 + w_2 x_2 + b = 0$.
This forms a line: $x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Create dense grid to visualize probability surface
x1_mesh, x2_mesh = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 200),
    np.linspace(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 200)
)
grid_pts = np.c_[x1_mesh.ravel(), x2_mesh.ravel()]
probs = log_reg.predict_proba(grid_pts).reshape(x1_mesh.shape)

# 1. Decision Boundary & Probability Contours
contour = ax1.contourf(x1_mesh, x2_mesh, probs, levels=20, cmap="coolwarm", alpha=0.6)
plt.colorbar(contour, ax=ax1, label="P(Y = 1 | X)")

# Plot the explicit boundary line (P = 0.5)
w1, w2 = log_reg.w[0, 0], log_reg.w[1, 0]
b_val = log_reg.b
x1_line = np.linspace(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 100)
x2_line = -(w1 * x1_line + b_val) / w2
ax1.plot(x1_line, x2_line, color="black", linewidth=2.5, linestyle="--", label="Decision Boundary (P=0.5)")

# Overlay samples
ax1.scatter(X0[:, 0], X0[:, 1], color="#1d4ed8", edgecolors="k", label="Class 0")
ax1.scatter(X1[:, 0], X1[:, 1], color="#b91c1c", edgecolors="k", label="Class 1")
ax1.set_title("Logistic Regression Decision Boundary", fontsize=13)
ax1.set_xlabel("Feature x_1")
ax1.set_ylabel("Feature x_2")
ax1.set_ylim(X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5)
ax1.legend()

# 2. BCE Loss Progression
ax2.plot(log_reg.loss_history, color="#7c3aed", linewidth=2)
ax2.set_title("Binary Cross-Entropy Loss Over Epochs", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("BCE Loss")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. PyTorch Verification

Let us confirm that our custom backward pass gradients match PyTorch autograd gradients on this exact batch.

In [ ]:
from nnlib.math import relative_error

# NumPy gradients
y_preds_np = log_reg.forward(X_2d)
grad_w_np, grad_b_np = log_reg.backward(X_2d, y_preds_np, y_2d)

# PyTorch autograd
X_pt = torch.tensor(X_2d, dtype=torch.float64)
y_pt = torch.tensor(y_2d, dtype=torch.float64)
w_pt = torch.tensor(log_reg.w.copy(), dtype=torch.float64, requires_grad=True)
b_pt = torch.tensor(float(log_reg.b), dtype=torch.float64, requires_grad=True)

z_pt = X_pt @ w_pt + b_pt
probs_pt = torch.sigmoid(z_pt)
loss_pt = torch.nn.functional.binary_cross_entropy(probs_pt, y_pt)
loss_pt.backward()

err_w = relative_error(grad_w_np, w_pt.grad.numpy())
err_b = relative_error(np.array([grad_b_np]), np.array([b_pt.grad.item()]))

print(f"Weight Gradient Relative Error: {err_w:.2e}")
print(f"Bias Gradient Relative Error:   {err_b:.2e}")
assert err_w < 1e-7 and err_b < 1e-7
print("PyTorch verification: 100% MATCH!")